# Initialize Prefect variable `processing-storage-configuration`

See: https://pforge-exchange2.astrium.eads.net/confluence/pages/viewpage.action?pageId=477103165

In [ ]:
# Imports
import json
import shutil
from IPython.display import Markdown
from pathlib import Path
from prefect.variables import Variable
from resources import utils

In [ ]:
var_name = "processing-storage-configuration"
try:
    existing_values = await Variable.get(var_name)
    dump = f"\n{json.dumps(dict(sorted(existing_values.items())), indent=2)}"
except AttributeError:
    existing_values = {}
    dump = "None"
print(f"Existing values for {var_name!r}: {dump}")

In [ ]:
# For the new values, start from the template file that is bundled with rs-client-libraries. 
# We copy it to the local folder with a .txt extension to open it with the text editor.
import rs_client
template_file_org = "storage_configuration.json"
template_file = template_file_org + ".txt"
shutil.copy(
    str(Path(rs_client.__file__).parent.parent / "config" / template_file_org), 
    template_file
)
!chmod u+w ./{template_file}

# Then display a markdown message with a link to it
relative_from_home = str(Path(template_file).resolve().relative_to(Path.home(), walk_up=True))
display(Markdown(f"""Please open and modify the template file: [{relative_from_home}]({template_file})

Then run the next Jupyter cell."""),
)

In [ ]:
# Read the modified template file
with open(str(template_file), encoding="utf-8") as f:
    new_values = json.load(f)

print(f"New values for {var_name!r}:\n{json.dumps(new_values, indent=2)}")

In [ ]:
# Overwrite values ?
diff = utils.compare_dict(existing_values, new_values)
if not diff:
    print("No differences with existing values.")
else:
    print(diff)
    answer = input(f"\nOverwrite these values in Prefect variable {var_name!r} (Y/n)?")
    
    if answer.lower() == "y":
        await Variable.set(var_name, new_values, overwrite=True)
        print(f"{var_name!r} overwritten.")
        existing_values = new_values